# Environment / Dependencies Setup
One command installs everything the lab needs:

In [ ]:
!pip install -qU langchain langchain-community langchain-openai langchain-huggingface sentence-transformers "torch>=2.5" langchain-text-splitters faiss-cpu python-dotenv

### Setup — Silence the Noisy Libraries
These libraries print progress bars and warnings that bury the actual output. This cell touches nothing in the pipeline; it just quiets things down so you can see your results.

In [ ]:
import logging
import os
import warnings

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

for noisy in ("transformers", "sentence_transformers", "huggingface_hub", "httpx"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

### Setup — Import the Tools
LangChain splits its helpers into small packages, one per job. The imports below are grouped by that job so you can see where everything comes from.

In [ ]:
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

### Setup — Load the API Key and Build the Two Models
This cell authenticates the LLM and loads the embedding model once, so every later step reuses the same objects instead of reloading them.

In [4]:
load_dotenv(".env")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Paste your OpenRouter API key: ").strip()

print("API key loaded.")

API key loaded.


Now build the LLM. `base_url` is what sends the request to OpenRouter rather than OpenAI — same client, different server.

In [4]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",  # a free model on OpenRouter
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0,  # 0 = as repeatable as the model allows
)

print("LLM ready:", llm.model_name)

LLM ready: nvidia/nemotron-3-super-120b-a12b:free


Then the embedding model. It runs on your own machine, so it costs nothing and needs no key; the first run downloads ~90 MB and then remembers it.

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model ready: 384 numbers per chunk")

Embedding model ready: 384 numbers per chunk


### Step 1 — Load the Document
This step turns the file into `Document` objects the rest of the pipeline can cut up.

In [5]:
# The path is relative, so Jupyter must be running from inside this lab folder.
DOC_PATH = "data/sample_text_document.txt"

# read_text() reads the whole file and hands back one long Python string.
raw_text = Path(DOC_PATH).read_text(encoding="utf-8")

# "\f" is the form-feed character -- the traditional ASCII page break in
# plain text. Splitting on it gives us one piece per page.
page_texts = [text.strip() for text in raw_text.split("\f") if text.strip()]

# Wrap each page in a Document: the text, plus a metadata dict recording
# where it came from. This metadata survives chunking, which is the point.
pages = [
    Document(page_content=text, metadata={"source": DOC_PATH, "page": number})
    for number, text in enumerate(page_texts)
]

total_characters = sum(len(page.page_content) for page in pages)

print(f"Loaded {len(pages)} page(s), {total_characters:,} characters.")

Loaded 2 page(s), 4,062 characters.


### Step 2 — Chunk the Text
This step slices the long text into small, overlapping pieces — the units the retriever will actually search.

In [6]:
# Each chunk aims to be ~400 characters long...
CHUNK_SIZE = 400
# ...and each new chunk repeats the last 50 characters of the previous one.
CHUNK_OVERLAP = 50

# Build the splitter with those two settings.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Apply it to every page we loaded in Step 1.
chunks = text_splitter.split_documents(pages)

print(f"Created {len(chunks)} chunks ({CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap).")
print("\nFirst chunk:\n" + chunks[0].page_content)

Created 13 chunks (400 chars, 50 overlap).

First chunk:
A Short Guide to Coastal Wetland Restoration
Synthetic Sample Document — Original Content, No Copyright Restrictions
1. Introduction
Coastal wetlands sit at the boundary between land and sea, absorbing storm surge, filtering runoff, and
providing nursery habitat for fish and shellfish. Over the past century, many of these ecosystems have


### Step 3 — Embed the Chunks and Build the Index
This is where text becomes searchable numbers. LangChain does the embedding *and* the storing in one call.

In [7]:
# from_documents does two things at once:
#   1. runs embedding_model over every chunk  ->  13 lists of 384 numbers
#   2. stores all 13 vectors in a FAISS index that similarity_search can query
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model,
    # Compare by INNER PRODUCT. Because every vector is unit length, the inner
    # product IS the cosine similarity, so the number comes back readable.
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)

print("Chunks indexed:", len(chunks))
print("Numbers per vector:", vector_store.index.d)
print("Index type:", type(vector_store.index).__name__)

Chunks indexed: 13
Numbers per vector: 384
Index type: IndexFlatIP


### Step 4 — Retrieve the Top-k Chunks
This step embeds your question and asks FAISS which chunks are closest to it.

In [8]:
# The question we want answered. Try changing it later.
QUESTION = "Why do restoration teams reintroduce tidal flow gradually instead of all at once?"

# How many chunks to bring back. 3 is a sensible starting point.
TOP_K = 3

# similarity_search_with_score turns your text into a vector and returns
# (chunk, score) pairs, best first.
matches = vector_store.similarity_search_with_score(QUESTION, k=TOP_K)

print(f"QUESTION: {QUESTION}\n")

for rank, (chunk, score) in enumerate(matches, start=1):
    # With MAX_INNER_PRODUCT the score is already the cosine similarity:
    # higher is better, 1 is identical. No conversion needed.
    print(f"--- Chunk {rank} | page {chunk.metadata['page']} | match {score:.3f} ---")
    print(chunk.page_content.strip() + "\n")

QUESTION: Why do restoration teams reintroduce tidal flow gradually instead of all at once?

--- Chunk 1 | page 0 | match 0.771 ---
3. Reintroducing Tidal Flow
Once barriers are addressed, restoration teams usually reintroduce tidal flow gradually rather than all at
once. A sudden full breach can erode unconsolidated fill material before vegetation has a chance to
stabilize the soil. Phased approaches might open a small channel first, monitor erosion and sediment

--- Chunk 2 | page 0 | match 0.626 ---
tide gates installed decades earlier for agricultural drainage. Removing or resizing these structures is often
the single most effective step in a restoration project, since it allows the natural rise and fall of the tide to
resume moving sediment, nutrients, and organisms in and out of the site.
3. Reintroducing Tidal Flow

--- Chunk 3 | page 0 | match 0.485 ---
been drained, filled, or cut off from tidal flow to make way for farmland, roads, and coastal development.
This guide summariz

### Step 5 — Generate the Answer
This step hands the retrieved chunks to the LLM and gets a grounded answer back.

In [9]:
# Glue the retrieved chunks into one block of text for the prompt.
context = "\n\n".join(chunk.page_content.strip() for chunk, _ in matches)

# The prompt template. {context} and {question} are holes we fill in below.
rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer is not in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

# The pipe operator chains three things together, left to right:
#   fill in the prompt  ->  send it to the LLM  ->  keep only the text
rag_chain = rag_prompt | llm | StrOutputParser()

# Run the chain by handing it exactly the names of the holes in the template.
answer = rag_chain.invoke({"context": context, "question": QUESTION})

print(f"Question: {QUESTION}")
print(f"\nAnswer: {answer}")

Question: Why do restoration teams reintroduce tidal flow gradually instead of all at once?

Answer: Restoration teams reintroduce tidal flow gradually because a sudden full breach can erode unconsolidated fill material before vegetation has a chance to stabilize the soil.


### Step 6 — Sweep the Two Settings That Matter
Now that the pipeline runs, this step changes the two knobs that decide how good it is, and prints what each one buys.

In [10]:
# --- Knob 1: TOP_K — how many chunks reach the LLM ---
# No LLM calls here, so this sweep is fast and free.
print("--- Changing TOP_K (how many chunks reach the LLM) ---")
for k in (1, 3, 5):
    trial_matches = vector_store.similarity_search_with_score(QUESTION, k=k)
    context_characters = sum(len(chunk.page_content) for chunk, _ in trial_matches)
    print(f"TOP_K={k}: {k} chunks, {context_characters:,} context chars, best match {trial_matches[0][1]:.3f}")

# --- Knob 2: CHUNK_SIZE — how finely we cut the document ---
print()
print("--- Changing CHUNK_SIZE (how finely we cut the document) ---")
for size in (300, 400, 800):
    # A different chunk size means different chunks, which means we must
    # rebuild the whole index before searching it again.
    trial_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    trial_chunks = trial_splitter.split_documents(pages)
    trial_store = FAISS.from_documents(
        documents=trial_chunks,
        embedding=embedding_model,
        distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
    )
    best_chunk, best_score = trial_store.similarity_search_with_score(QUESTION, k=1)[0]
    # Show the first few words of whatever won, so we can see *which* section it was.
    opening_words = " ".join(best_chunk.page_content.split()[:8])
    print(f"CHUNK_SIZE={size}: {len(trial_chunks):>2} chunks, best match {best_score:.3f} -> \"{opening_words}...\"")

--- Changing TOP_K (how many chunks reach the LLM) ---
TOP_K=1: 1 chunks, 335 context chars, best match 0.771
TOP_K=3: 3 chunks, 965 context chars, best match 0.771
TOP_K=5: 5 chunks, 1,604 context chars, best match 0.771

--- Changing CHUNK_SIZE (how finely we cut the document) ---
CHUNK_SIZE=300: 18 chunks, best match 0.814 -> "3. Reintroducing Tidal Flow Once barriers are addressed,..."
CHUNK_SIZE=400: 13 chunks, best match 0.771 -> "3. Reintroducing Tidal Flow Once barriers are addressed,..."


CHUNK_SIZE=800:  6 chunks, best match 0.687 -> "3. Reintroducing Tidal Flow Once barriers are addressed,..."


### Step 7 — Ask a Question the Document Cannot Answer
Step 4 gave us three chunks that all looked fine. This step finds out what "not fine" actually looks like, by asking about something this document has no opinion on.

In [11]:
# A question with no possible answer in a wetland-restoration guide.
OFF_TOPIC_QUESTION = "Who won the 1998 FIFA World Cup?"

off_topic_matches = vector_store.similarity_search_with_score(OFF_TOPIC_QUESTION, k=1)
off_topic_score = off_topic_matches[0][1]

print(f"off-topic best match: {off_topic_score:.3f}")
print(f"on-topic best match:  {matches[0][1]:.3f}")

off-topic best match: 0.021
on-topic best match:  0.771
